# Thermoelectric Seebeck datasets — first look

This notebook loads three datasets discussed for the thesis (Seebeck coefficient prediction via symbolic
regression, benchmarked against the single parabolic band / SPB model) and runs basic exploration on each:

1. **DFT/BoltzTraP (Ricci et al. 2017)** — via `matminer`'s `boltztrap_mp` dataset (8,924 compounds,
   single point: 300 K, n = 1e18 cm⁻³). This is the easy-to-load, matminer-hosted slice of Ricci's data.
   Note up front: this is **not** the full continuous S(n,T) surface described in the original paper
   (which spans 10¹⁶–10²⁰ cm⁻³ and 100–1300 K) — it's a single-point summary. We check what we actually
   got in Section 1.4, and Section 5 explains how to get the fuller dataset if the single point isn't enough.
2. **teMatDb272** (Ryu et al. 2025) — real experimental T-dependent Seebeck curves, pulled live from its
   public GitHub repo.
3. **ChemDataExtractor thermoelectric DB** (Sierepeklis & Cole 2022) — auto-extracted experimental data,
   pulled live from Figshare via its API.

Goal of this notebook: **understand what's actually in each dataset** (schema, size, missing fields —
especially carrier concentration — and family/grouping metadata) and do a first honest check of how much
composition overlap exists between the DFT set and the two experimental sets, since that overlap is what
determines whether a matched DFT-vs-experiment comparison is realistic at scale or only as a handful of
case studies.

Run cells top to bottom. Everything downloads live — no files need to be uploaded.


## 0. Setup


In [1]:
!pip install -q matminer pymatgen requests pandas

import requests
import pandas as pd
import numpy as np
import io
import re
import json
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 160)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 1.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 71.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.1/829.1 kB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.6/62.6 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.3/332.3 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 962.5/962.5 kB 38.3 MB/s eta 0:00:00
   ━━━━━━

## 1. DFT / BoltzTraP data (Ricci et al. 2017), via matminer

`matminer` ships a loader for `boltztrap_mp`, which downloads and caches the dataset automatically the
first time you call it (may take a minute).


In [2]:
from matminer.datasets import load_dataset, get_available_datasets

# Sanity check the dataset is really in matminer's catalog before pulling it
available = get_available_datasets()
print('boltztrap_mp' in available, '-> boltztrap_mp is available in this matminer install')

df_dft = load_dataset('boltztrap_mp')
print('Shape:', df_dft.shape)
df_dft.head()


boltztrap_mp: Effective mass and thermoelectric properties of 8924 compounds in The  Materials Project database that are calculated by the BoltzTraP software package run on the GGA-PBE or GGA+U density functional theory calculation results. The properties are reported at the temperature of 300 Kelvin and the carrier concentration of 1e18 1/cm3.

brgoch_superhard_training: 2574 materials used for training regressors that predict shear and bulk modulus.

castelli_perovskites: 18,928 perovskites generated with ABX combinatorics, calculating gllbsc band gap and pbe structure, and also reporting absolute band edge positions and heat of formation.

citrine_thermal_conductivity: Thermal conductivity of 872 compounds measured experimentally and retrieved from Citrine database from various references. The reported values are measured at various temperatures of which 295 are at room temperature.

dielectric_constant: 1,056 structures with dielectric properties, calculated with DFPT-PBE.

double_

Fetching https://ndownloader.figshare.com/files/13297511 in MB: 14.020608MB [00:00, 383.02MB/s]                         


Shape: (8924, 9)


,mpid,pf_n,pf_p,s_n,s_p,formula,m_n,m_p,structure
0,mp-10070,0.865,0.0125,-421.0,760.0,BaAg(PO3)3,0.578,72.800,"[[ 0.08245398 10.58009491 11.61923254] O, [3.1..."
1,mp-10086,1.050,0.6440,-393.0,567.0,YSF,0.541,4.020,"[[2.84699546 0.94899849 0. ] F, [0.9489..."
2,mp-10096,0.977,0.4200,-305.0,471.0,Na3Sr3GaP4,0.270,1.020,"[[-2.8085287 7.06608376 1.25800196] Na, [1...."
3,mp-10103,0.889,0.0257,-416.0,871.0,YbLiAlF6,0.548,20.300,"[[2.52287112 1.45658029 7.18290524] Li, [-2.85..."
4,mp-10155,0.719,1.5700,-496.0,510.0,P2Ir,1.160,0.667,"[[0.86371064 5.14343422 4.80450637] P, [4.5512..."


### 1.1 Schema and missingness


In [3]:
print(df_dft.dtypes)
print()
print('Missing values per column:')
print(df_dft.isna().sum())


mpid          object
pf_n         float64
pf_p         float64
s_n          float64
s_p          float64
formula       object
m_n          float64
m_p          float64
structure     object
dtype: object

Missing values per column:
mpid         0
pf_n         0
pf_p         0
s_n          0
s_p          0
formula      0
m_n          0
m_p          0
structure    0
dtype: int64


### 1.2 Basic statistics on the transport properties


In [4]:
cols_of_interest = [c for c in ['s_n', 's_p', 'pf_n', 'pf_p', 'm_n', 'm_p'] if c in df_dft.columns]
df_dft[cols_of_interest].describe()


,s_n,s_p,pf_n,pf_p,m_n,m_p
count,8924.000000,8924.000000,8.924000e+03,8924.000000,8.924000e+03,8.924000e+03
mean,-545.180536,676.105806,6.662679e-01,0.299013,1.483161e+04,2.022671e+04
std,163.648598,142.910040,4.668595e-01,0.434866,7.967115e+05,1.235763e+06
min,-1120.000000,-691.000000,8.930000e-07,0.000015,1.590000e-02,1.600000e-02
25%,-676.000000,636.000000,2.670000e-01,0.064100,5.767500e-01,3.710000e+00
50%,-549.000000,703.000000,7.100000e-01,0.166000,1.790000e+00,1.100000e+01
75%,-418.000000,756.000000,9.360000e-01,0.373000,1.380000e+01,4.172500e+01
max,655.000000,1090.000000,6.710000e+00,7.290000,6.650000e+07,1.140000e+08


### 1.3 How many unique formulas / compounds

This matters directly for the DFT-vs-experiment matching question: Ricci/Materials Project entries are
almost all *ordered, near-stoichiometric* compounds, not doped/alloyed variants — check that assumption
here.


In [5]:
print('Unique formulas:', df_dft['formula'].nunique(), 'out of', len(df_dft), 'rows')
print()
print('A few example formulas (look for doping/fractional-stoichiometry notation vs clean formulas):')
print(df_dft['formula'].sample(15, random_state=0).tolist())


Unique formulas: 7380 out of 8924 rows

A few example formulas (look for doping/fractional-stoichiometry notation vs clean formulas):
['LaH6(ClO3)3', 'RbI3', 'Tl2SeO4', 'Tm2O3', 'ScCrO3', 'Cd2GaAgS4', 'Li2Fe(PO4)2', 'Na2SnO3', 'MnP2O7', 'Y2Sn2O7', 'TiAs2O7', 'H2C2S3N2(OF)6', 'P6(Pb2O5)5', 'SiS2', 'Sr7GeN6']


### 1.4 Confirm the single-point limitation

The original Ricci et al. 2017 paper reports Seebeck across a doping grid (10¹⁶–10²⁰ cm⁻³) and a
temperature grid (100–1300 K). Check whether this matminer slice actually varies by n and T, or whether
every row really is fixed at 300 K / 1e18 cm⁻³ as documented.


In [6]:
# If there's no explicit temperature or doping column, that itself confirms the single-point nature.
possible_condition_cols = [
    c for c in df_dft.columns
    if c.lower() in ('t', 'temp', 'temperature', 'n', 'doping', 'carrier_concentration', 'carrier_conc')
]
print('Condition-like columns found:', possible_condition_cols)
print('All columns:', list(df_dft.columns))


Condition-like columns found: []
All columns: ['mpid', 'pf_n', 'pf_p', 's_n', 's_p', 'formula', 'm_n', 'm_p', 'structure']


## 2. teMatDb272 (Ryu et al. 2025) — real experimental data

Pulled live from the public GitHub repo `byungkiryu/teMatDb`. We list the directory via the GitHub API
first rather than guessing filenames, then download whatever CSVs are actually there.


In [7]:
repo_api = 'https://api.github.com/repos/byungkiryu/teMatDb/contents/teMatDb_publication'
resp = requests.get(repo_api, timeout=30)
print('Status:', resp.status_code)
listing = resp.json()
if isinstance(listing, list):
    for f in listing:
        print(f['name'], '-', f.get('size'), 'bytes ->', f['download_url'])
else:
    print(listing)  # will show an error/rate-limit message if the API call failed


Status: 200
README.md - 1350 bytes -> https://raw.githubusercontent.com/byungkiryu/teMatDb/main/teMatDb_publication/README.md
generate_teMatDb272_from_teMatDbv116.py - 11004 bytes -> https://raw.githubusercontent.com/byungkiryu/teMatDb/main/teMatDb_publication/generate_teMatDb272_from_teMatDbv116.py
teMatDb272_dataset_20250515.zip - 2282543 bytes -> https://raw.githubusercontent.com/byungkiryu/teMatDb/main/teMatDb_publication/teMatDb272_dataset_20250515.zip
teMatDb272_dataset_20250515 - 0 bytes -> None


If the cell above didn't return a file list (e.g. GitHub rate-limited the request), browse
`https://github.com/byungkiryu/teMatDb/tree/main/teMatDb_publication` manually in a browser tab, note the
CSV filenames, and paste them into the `csv_urls` dict below instead of relying on auto-discovery.


In [8]:
csv_urls = {
    f['name']: f['download_url']
    for f in listing
    if isinstance(listing, list) and f['name'].endswith('.csv')
}
print(csv_urls)

tematdb_tables = {}
for name, url in csv_urls.items():
    try:
        tematdb_tables[name] = pd.read_csv(url)
        print(name, '->', tematdb_tables[name].shape)
    except Exception as e:
        print(name, 'FAILED:', e)


{}


### 2.1 Inspect the sample-level metadata table (composition, family, processing info)


In [9]:
# Try to find the samples/metadata table among whatever got loaded
samples_key = next((k for k in tematdb_tables if 'sample' in k.lower()), None)
print('Using table:', samples_key)
df_samples = tematdb_tables.get(samples_key)
if df_samples is not None:
    print(df_samples.shape)
    print(df_samples.columns.tolist())
    display(df_samples.head())


Using table: None


### 2.2 Confirm whether carrier concentration is actually present (expected: no)


In [10]:
if df_samples is not None:
    carrier_like = [c for c in df_samples.columns if re.search(r'carrier|hall|doping|conc', c, re.I)]
    print('Carrier-concentration-like columns found:', carrier_like)


### 2.3 Family / group metadata and processing route


In [11]:
if df_samples is not None:
    for col in ['GROUP', 'BASEMAT', 'SINTERING']:
        if col in df_samples.columns:
            print(f'--- {col} ---')
            print(df_samples[col].value_counts().head(20))
            print()


### 2.4 Inspect the raw/collocated TEP (property-vs-temperature) tables


In [12]:
tep_keys = [k for k in tematdb_tables if k != samples_key]
for k in tep_keys:
    print('---', k, '---')
    print(tematdb_tables[k].shape)
    display(tematdb_tables[k].head())


### 2.5 How many temperature points per sample (curve-fitting the SPB model needs enough points per curve)


In [13]:
# Adjust column names below once you've seen the actual schema from 2.4
for k in tep_keys:
    df_t = tematdb_tables[k]
    id_col = next((c for c in df_t.columns if 'sample' in c.lower() or c.lower() == 'id'), None)
    if id_col:
        counts = df_t.groupby(id_col).size()
        print(k, '-> points per sample: min', counts.min(), 'median', counts.median(), 'max', counts.max())


## 3. ChemDataExtractor thermoelectric database (Sierepeklis & Cole 2022)

Pulled live from Figshare via its public API (article ID 19658787), so we discover the real file
names/URLs rather than guessing.


In [14]:
fig_api = 'https://api.figshare.com/v2/articles/19658787'
resp = requests.get(fig_api, timeout=30)
print('Status:', resp.status_code)
article = resp.json()
files = article.get('files', [])
for f in files:
    print(f['name'], '-', f.get('size'), 'bytes ->', f['download_url'])


Status: 200
thermoelectricsdb-main.zip - 5411046 bytes -> https://ndownloader.figshare.com/files/34915236
TE_databases.zip - 10665264 bytes -> https://ndownloader.figshare.com/files/36554916


In [15]:
csv_file = next((f for f in files if f['name'].lower().endswith('.csv')), None)
print('Using:', csv_file['name'] if csv_file else None)

df_cde = None
if csv_file:
    r = requests.get(csv_file['download_url'], timeout=60)
    df_cde = pd.read_csv(io.BytesIO(r.content))
    print(df_cde.shape)
    display(df_cde.head())
else:
    print('No CSV found directly on the article -- check the printed file list above;',
          'the data may be inside a zip, in which case download+unzip it instead.')


Using: None
No CSV found directly on the article -- check the printed file list above; the data may be inside a zip, in which case download+unzip it instead.


### 3.1 Schema and missingness


In [16]:
if df_cde is not None:
    print(df_cde.dtypes)
    print()
    print(df_cde.isna().sum())


### 3.2 Confirm whether carrier concentration is present (expected: no)


In [17]:
if df_cde is not None:
    carrier_like = [c for c in df_cde.columns if re.search(r'carrier|hall|doping|conc', c, re.I)]
    print('Carrier-concentration-like columns found:', carrier_like)


### 3.3 Property coverage (Seebeck, conductivity, kappa, PF, ZT) and value ranges


In [18]:
specifier_col = None
value_col = None
if df_cde is not None:
    specifier_col = next((c for c in df_cde.columns if c.lower() == 'specifier'), None)
    if specifier_col:
        print(df_cde[specifier_col].value_counts())
    value_col = next((c for c in df_cde.columns if c.lower() == 'value'), None)
    if value_col:
        print(df_cde[value_col].describe())


### 3.4 Host/dopant parsing quality (needed for any composition-family grouping)


In [19]:
resolution_col = None
name_col = None
if df_cde is not None:
    resolution_col = next((c for c in df_cde.columns if 'resolution' in c.lower()), None)
    name_col = next((c for c in df_cde.columns if c.lower() == 'name'), None)
    print('Resolution column:', resolution_col, '| Name column:', name_col)
    if name_col:
        sample_n = min(15, df_cde[name_col].notna().sum())
        print(df_cde[name_col].dropna().sample(sample_n, random_state=0).tolist())


## 4. Composition-overlap check: is DFT-vs-experiment matching realistic at scale?

This is the practical question from the discussion: normalize formulas on both sides and count how many
host compositions genuinely appear in **both** the DFT set and each experimental set. This tells us
whether a database-scale matched comparison is viable, or whether we should only expect a handful of
well-known systems (PbTe, Bi2Te3, etc.) to overlap — in which case the plan is a small, manually-curated
case-study set, not an automated join.


In [20]:
from pymatgen.core.composition import Composition

def to_reduced_formula(s):
    """Best-effort normalization to a comparable reduced formula. Returns None on failure
    (e.g. text that isn't a clean chemical formula, which is common in messy experimental fields)."""
    if not isinstance(s, str) or not s.strip():
        return None
    try:
        return Composition(s).reduced_formula
    except Exception:
        return None

dft_formulas = set(df_dft['formula'].map(to_reduced_formula).dropna())
print('DFT set: reduced formulas parsed:', len(dft_formulas),
      'out of', df_dft['formula'].nunique(), 'unique raw formulas')


/usr/lib/python3.13/functools.py:1025: UserWarning: No Pauling electronegativity for Ar. Setting to NaN. This has no physical meaning, and is mainly done to avoid errors caused by the code expecting a float.
  val = self.func(instance)


DFT set: reduced formulas parsed: 7380 out of 7380 unique raw formulas


In [21]:
# --- teMatDb272 side: use BASEMAT if available, since it's already meant to represent the host matrix ---
tematdb_formulas = set()
if df_samples is not None and 'BASEMAT' in df_samples.columns:
    tematdb_formulas = set(df_samples['BASEMAT'].map(to_reduced_formula).dropna())
    print('teMatDb272: BASEMAT reduced formulas parsed:', len(tematdb_formulas),
          'out of', df_samples['BASEMAT'].nunique(), 'unique raw BASEMAT values')

overlap_tematdb = dft_formulas & tematdb_formulas
print()
print('Overlap DFT <-> teMatDb272 host formulas:', len(overlap_tematdb))
print(sorted(overlap_tematdb)[:40])



Overlap DFT <-> teMatDb272 host formulas: 0
[]


In [22]:
# --- ChemDataExtractor side: try the parsed host from Resolution, fall back to raw Name ---
def extract_host(s):
    if not isinstance(s, str):
        return None
    m = re.search(r"host['\"]?\s*[:=]\s*['\"]?([A-Za-z0-9().]+)", s)
    return m.group(1) if m else None

cde_formulas = set()
if df_cde is not None:
    host_series = None
    if resolution_col and df_cde[resolution_col].notna().any():
        # Resolution is typically stored as a stringified dict; try to pull out a 'host' key
        host_series = df_cde[resolution_col].map(extract_host)
    if host_series is None or host_series.notna().sum() == 0:
        host_series = df_cde[name_col]
    cde_formulas = set(host_series.map(to_reduced_formula).dropna())
    print('ChemDataExtractor: reduced host formulas parsed:', len(cde_formulas))

overlap_cde = dft_formulas & cde_formulas
print()
print('Overlap DFT <-> ChemDataExtractor host formulas:', len(overlap_cde))
print(sorted(overlap_cde)[:40])



Overlap DFT <-> ChemDataExtractor host formulas: 0
[]


### 4.1 Row counts behind the overlapping formulas (not just unique-formula counts)


In [23]:
# How many actual experimental data points do the overlapping systems represent?
# This is the number that determines whether a matched subset is usable, not just the formula count.
if df_samples is not None and overlap_tematdb:
    mask = df_samples['BASEMAT'].map(to_reduced_formula).isin(overlap_tematdb)
    print('teMatDb272 samples belonging to an overlapping host formula:', mask.sum(), '/', len(df_samples))
    print(df_samples.loc[mask, 'BASEMAT'].value_counts().head(20))


## 5. Summary — what to actually look at once this has run

Fill this in after running the cells above:

- **DFT set size / coverage**: rows in `df_dft`, and whether Section 1.4 confirms it's single-point only.
  If it is (expected), the full continuous S(n,T) surface needs to come from the original Dryad deposit
  (Ricci et al. 2017, DOI `10.1038/sdata.2017.85`; dataset itself at Dryad DOI `10.5061/dryad.gn001`) or
  reconstructed via `pymatgen`'s `BoltztrapRunner`/`MPRester` against Materials Project directly — not
  from this quick matminer slice.
- **teMatDb272 size / schema**: confirm no carrier-concentration column (Section 2.2), see the real
  `GROUP`/`BASEMAT`/`SINTERING` distributions (2.3), and check points-per-sample (2.5) — SPB curve-fitting
  needs a reasonable number of temperature points per curve to be numerically stable.
- **ChemDataExtractor DB size / schema**: confirm no carrier-concentration column (3.2), check how cleanly
  `Resolution`/`Name` actually parses into host formulas (3.4) — this determines whether the matching in
  Section 4 is trustworthy or needs better parsing.
- **Overlap counts (Section 4)**: this is the number that answers the open question from the discussion —
  if it's a handful of well-known systems (PbTe, Bi2Te3, skutterudites, half-Heuslers), that confirms the
  "small manually-curated case-study set" plan rather than a database-scale automated join.

Next step once this is reviewed: decide whether to pull the fuller multi-point Ricci/BoltzTraP dataset
(needed for the true SPB residual across the doping/temperature grid), and whether the overlap found here
is enough to build a real matched validation subset or should stay a smaller case-study check.
